In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.constants import precision

In [2]:
df = pd.read_csv('C:/Users/Varad Vankar/Downloads/global_inflation_data.csv')
print(df.columns.tolist())

['country_name', 'indicator_name', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [3]:
print(df)

           country_name                                   indicator_name  \
0           Afghanistan  Annual average inflation (consumer prices) rate   
1               Albania  Annual average inflation (consumer prices) rate   
2               Algeria  Annual average inflation (consumer prices) rate   
3               Andorra  Annual average inflation (consumer prices) rate   
4                Angola  Annual average inflation (consumer prices) rate   
..                  ...                                              ...   
191             Vietnam  Annual average inflation (consumer prices) rate   
192  West Bank and Gaza  Annual average inflation (consumer prices) rate   
193               Yemen  Annual average inflation (consumer prices) rate   
194              Zambia  Annual average inflation (consumer prices) rate   
195            Zimbabwe  Annual average inflation (consumer prices) rate   

     1980  1981  1982  1983  1984  1985   1986   1987  ...   2015   2016  \
0    13.4  

In [4]:
df.describe()
df.sum

<bound method DataFrame.sum of            country_name                                   indicator_name  \
0           Afghanistan  Annual average inflation (consumer prices) rate   
1               Albania  Annual average inflation (consumer prices) rate   
2               Algeria  Annual average inflation (consumer prices) rate   
3               Andorra  Annual average inflation (consumer prices) rate   
4                Angola  Annual average inflation (consumer prices) rate   
..                  ...                                              ...   
191             Vietnam  Annual average inflation (consumer prices) rate   
192  West Bank and Gaza  Annual average inflation (consumer prices) rate   
193               Yemen  Annual average inflation (consumer prices) rate   
194              Zambia  Annual average inflation (consumer prices) rate   
195            Zimbabwe  Annual average inflation (consumer prices) rate   

     1980  1981  1982  1983  1984  1985   1986   1987  .

In [5]:
df.shape[0]

196

In [8]:
df['country_name'].nunique()

196

In [7]:
df['country_name'].value_counts()


country_name
Afghanistan           1
Albania               1
Algeria               1
Andorra               1
Angola                1
                     ..
Vietnam               1
West Bank and Gaza    1
Yemen                 1
Zambia                1
Zimbabwe              1
Name: count, Length: 196, dtype: int64

In [4]:

id_cols  = ["country_name","indicator_name"]
year_cols = [col for col in df.columns if col not in id_cols]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='year',
    value_name='inflation_rate'
)
df_long['year'] = df_long['year'].astype(int)

df_long["inflation_rate"] = pd.to_numeric(df_long['inflation_rate'], errors="coerce")

df_long = df_long.drop(columns = ["indicator_name"])

df_long = df_long.sort_values(["country_name", "year"]).reset_index(drop=True)

print('\nlong format shape:',df_long.shape)
print(df_long.head(10))
print("\nMissing values:", df_long["inflation_rate"].isna().sum(), "out of", len(df_long))
print("\nYear range:", df_long["year"].min(), "-", df_long["year"].max())
print("Number of countries:", df_long["country_name"].nunique())


long format shape: (8820, 3)
  country_name  year  inflation_rate
0  Afghanistan  1980            13.4
1  Afghanistan  1981            22.2
2  Afghanistan  1982            18.2
3  Afghanistan  1983            15.9
4  Afghanistan  1984            20.4
5  Afghanistan  1985             8.7
6  Afghanistan  1986            -2.1
7  Afghanistan  1987            18.4
8  Afghanistan  1988            27.5
9  Afghanistan  1989            71.5

Missing values: 868 out of 8820

Year range: 1980 - 2024
Number of countries: 196


In [5]:
#setting up threshold to identify
X = 50
df_long["is_hyperinflation"] = (df_long["inflation_rate"]>X).astype(int)

print(df_long["is_hyperinflation"].value_counts())
print(df_long["is_hyperinflation"].value_counts(normalize=True)*100)

is_hyperinflation
0    8460
1     360
Name: count, dtype: int64
is_hyperinflation
0    95.918367
1     4.081633
Name: proportion, dtype: float64


In [6]:
hyper_countries = df_long[df_long["is_hyperinflation"]==1]["country_name"].nunique()
print("Countries with at least one hyperinflation:", hyper_countries)

print(df_long[df_long["is_hyperinflation"]==1]["country_name"].value_counts().head(10))

print(df_long[df_long["is_hyperinflation"]==1]["year"].describe())

Countries with at least one hyperinflation: 63
country_name
Türkiye, Republic of       18
Congo, Dem. Rep. of the    16
Brazil                     16
Sudan                      16
Venezuela                  15
Belarus                    13
Angola                     13
Peru                       13
Uruguay                    11
Lebanon                    11
Name: count, dtype: int64
count     360.000000
mean     1994.258333
std        10.966774
min      1980.000000
25%      1987.000000
50%      1992.000000
75%      1996.000000
max      2024.000000
Name: year, dtype: float64


In [7]:
print(df_long[(df_long["is_hyperinflation"] == 1) & (df_long["year"] >= 2015)].shape[0])
print(df_long[(df_long["is_hyperinflation"] == 1) & (df_long["year"] >= 2010)].shape[0])

41
44


In [8]:
df_long = df_long.sort_values(["country_name", "year"]).reset_index(drop=True)
grp = df_long.groupby("country_name")["inflation_rate"]

df_long["lag_1"] = grp.shift(1)
df_long["lag_2"] = grp.shift(2)
df_long["lag_3"] = grp.shift(3)

df_long["roll_mean_3"] = grp.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df_long["roll_std_3"] = grp.shift(1).rolling(3).std().reset_index(level=0,drop=True)
df_long["roll_mean_5"] =grp.shift(1).rolling(5).mean().reset_index(level=0,drop=True)

df_long["yoy_change"] = df_long["lag_1"] - df_long["lag_2"]

df_long["high_inflation_count_5"] = (
    grp.shift(1).rolling(5).apply(lambda x: (x>20).sum())
    .reset_index(level=0, drop=True)
)

print(df_long.head(15))
print("\nMissing values per column:")
print(df_long.isna().sum())

   country_name  year  inflation_rate  is_hyperinflation  lag_1  lag_2  lag_3  \
0   Afghanistan  1980           13.40                  0    NaN    NaN    NaN   
1   Afghanistan  1981           22.20                  0  13.40    NaN    NaN   
2   Afghanistan  1982           18.20                  0  22.20  13.40    NaN   
3   Afghanistan  1983           15.90                  0  18.20  22.20   13.4   
4   Afghanistan  1984           20.40                  0  15.90  18.20   22.2   
5   Afghanistan  1985            8.70                  0  20.40  15.90   18.2   
6   Afghanistan  1986           -2.10                  0   8.70  20.40   15.9   
7   Afghanistan  1987           18.40                  0  -2.10   8.70   20.4   
8   Afghanistan  1988           27.50                  0  18.40  -2.10    8.7   
9   Afghanistan  1989           71.50                  1  27.50  18.40   -2.1   
10  Afghanistan  1990           47.40                  0  71.50  27.50   18.4   
11  Afghanistan  1991       

In [9]:
feature_cols = ["lag_1", "lag_2", "lag_3", "roll_mean_3", "roll_std_3",
                "roll_mean_5", "yoy_change", "high_inflation_count_5"]

df_model = df_long.dropna(subset=["inflation_rate"] + feature_cols).reset_index(drop=True)

print("Original rows:", len(df_long))
print("Rows after dropping NaNs:", len(df_model))
print("\nClass balance after dropping:")
print(df_model["is_hyperinflation"].value_counts())
print(df_model["is_hyperinflation"].value_counts(normalize=True)*100)

Original rows: 8820
Rows after dropping NaNs: 6967

Class balance after dropping:
is_hyperinflation
0    6725
1     242
Name: count, dtype: int64
is_hyperinflation
0    96.526482
1     3.473518
Name: proportion, dtype: float64


In [10]:
train = df_model[df_model["year"] < 2015].copy()
test = df_model[df_model["year"] >= 2015].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain class balance:")
print(train["is_hyperinflation"].value_counts())

print("\nTest class balance:")
print(test["is_hyperinflation"].value_counts())

Train shape: (5043, 12)
Test shape: (1924, 12)

Train class balance:
is_hyperinflation
0    4839
1     204
Name: count, dtype: int64

Test class balance:
is_hyperinflation
0    1886
1      38
Name: count, dtype: int64


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score
)

In [12]:
feature_cols = ["lag_1", "lag_2", "lag_3", "roll_mean_3", "roll_std_3",
                "roll_mean_5", "yoy_change", "high_inflation_count_5"]

x_train = train[feature_cols]
y_train = train["is_hyperinflation"]
x_test = test[feature_cols]
y_test = test["is_hyperinflation"]

neg = (y_train==0).sum()
pos = (y_train==0).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

#Train baseline model
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42
)
model.fit(x_train, y_train)

# Predictions
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAverage Precision (PR-AUC):", average_precision_score(y_test, y_proba))


scale_pos_weight: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1886
           1       0.88      0.61      0.72        38

    accuracy                           0.99      1924
   macro avg       0.94      0.80      0.86      1924
weighted avg       0.99      0.99      0.99      1924


Confusion Matrix:
[[1883    3]
 [  15   23]]

Average Precision (PR-AUC): 0.824715791822207


In [13]:
print(y_train.value_counts())
print(y_train.dtype)

is_hyperinflation
0    4839
1     204
Name: count, dtype: int64
int64


In [14]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print("neg:", neg, "pos:", pos)
print("scale_pos_weight:", scale_pos_weight)

neg: 4839 pos: 204
scale_pos_weight: 23.720588235294116


In [16]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAverage Precision (PR-AUC):", average_precision_score(y_test, y_proba))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1886
           1       0.62      0.84      0.71        38

    accuracy                           0.99      1924
   macro avg       0.81      0.92      0.85      1924
weighted avg       0.99      0.99      0.99      1924


Confusion Matrix:
[[1866   20]
 [   6   32]]

Average Precision (PR-AUC): 0.8305160184009285


In [17]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

lag_1                     0.668637
roll_mean_5               0.077465
yoy_change                0.063366
roll_mean_3               0.049572
roll_std_3                0.048879
lag_2                     0.035844
lag_3                     0.034776
high_inflation_count_5    0.021461
dtype: float32


In [21]:
y_pred = model.predict(x_test)
y_pred

array([0, 0, 0, ..., 1, 1, 1], shape=(1924,))

In [22]:
results = test[["country_name", "year", "inflation_rate", "is_hyperinflation"]].copy()
results["predicted"] = y_pred
results["predicted_probability"] = y_proba

# Show only the interesting cases: where the model predicted hyperinflation, or where it actually happened
interesting = results[(results["is_hyperinflation"] == 1) | (results["predicted"] == 1)]
print(interesting.sort_values("year"))

                  country_name  year  inflation_rate  is_hyperinflation  \
6786                 Venezuela  2015           121.7                  1   
5893                  Suriname  2016            55.5                  1   
6787                 Venezuela  2016           254.9                  1   
5736  South Sudan, Republic of  2017           213.0                  1   
5894                  Suriname  2017            22.0                  0   
6788                 Venezuela  2017           438.1                  1   
3637                     Libya  2018            14.0                  0   
5855                     Sudan  2018            63.3                  1   
6789                 Venezuela  2018         65374.1                  1   
5737  South Sudan, Republic of  2018            83.4                  1   
5738  South Sudan, Republic of  2019            49.3                  0   
6790                 Venezuela  2019         19906.0                  1   
542                   Bar